In [ ]:
!pip install dowhy
!pip install networkx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 865.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.5/245.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 60.9 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: cvxpy
    Found existing installation: cvxpy 1.6.7
    Uninstalling cvxpy-1.6.7:
      Successfully uninstalled cvxpy-1.6.7


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
project_path = "/content/drive/MyDrive/Colab Notebooks/CD Skripsi"

In [ ]:
import networkx as nx
from dowhy import gcm
import pandas as pd
import numpy as np

#DoWhy

In [ ]:
data_customer = pd.read_csv(
    '/content/drive/MyDrive/Colab Notebooks/CD Skripsi/CustomerCausal_dataset.csv',
    sep=',',
    engine='python'
)

data_customer.head()

,cancellation,no_delivery,delivery_delay,delivery_time,late_delivery_flag,avg_installments,total_payment,freight_value,product_description_length,product_photos_qty,is_dissatisfied
0,0.0,0.0,-5.0,6.0,0.0,8.0,141.90,12.00,236.0,1.0,0
1,0.0,0.0,-5.0,3.0,0.0,1.0,27.19,8.29,635.0,1.0,0
2,0.0,0.0,-2.0,25.0,0.0,8.0,86.22,17.22,177.0,3.0,0
3,0.0,0.0,-12.0,20.0,0.0,4.0,43.62,17.63,1741.0,5.0,0
4,0.0,0.0,-8.0,13.0,0.0,6.0,196.89,16.89,794.0,3.0,0


In [ ]:
graph = nx.DiGraph()
graph.add_edges_from([
    ('delivery_delay', 'no_delivery'),
        ('delivery_time', 'is_dissatisfied'),
        ('delivery_time', 'late_delivery_flag'),
        ('freight_value', 'delivery_time'),
        ('freight_value', 'is_dissatisfied'),
        ('freight_value', 'late_delivery_flag'),
        ('freight_value', 'no_delivery'),
        ('no_delivery', 'is_dissatisfied'),
        ('total_payment', 'delivery_time'),

        ('delivery_delay', 'late_delivery_flag'),
        ('delivery_time', 'no_delivery'),
        ('freight_value', 'delivery_delay'),
        ('late_delivery_flag', 'is_dissatisfied'),
        ('no_delivery', 'cancellation'),

        ('product_description_length', 'freight_value'),
        ('product_photos_qty', 'freight_value'),
        ('product_description_length', 'total_payment'),

        ('product_description_length', 'avg_installments'),
        ('total_payment', 'delivery_delay'),
        ('product_photos_qty', 'delivery_time'),
        ('avg_installments', 'delivery_time'),
        ('product_photos_qty', 'no_delivery'),

        ('avg_installments', 'delivery_delay'),
        ('total_payment', 'late_delivery_flag'),
        ('product_description_length', 'is_dissatisfied')
])

In [ ]:
causal_model = gcm.StructuralCausalModel(graph)

In [ ]:
gcm.auto.assign_causal_mechanisms(causal_model, data_customer)

In [ ]:
gcm.fit(causal_model, data_customer)

Fitting causal mechanism of node avg_installments: 100%|██████████| 11/11 [00:11<00:00,  1.07s/it]


In [ ]:
validation_summary = gcm.evaluate_causal_model(causal_model, data_customer)
validation_summary

Test permutations of given graph: 100%|██████████| 50/50 [03:23<00:00,  4.07s/it]


CausalModelEvaluationResult(mechanism_performances={'product_description_length': MechanismPerformanceResult(), 'product_photos_qty': MechanismPerformanceResult(), 'total_payment': MechanismPerformanceResult(), 'avg_installments': MechanismPerformanceResult(), 'freight_value': MechanismPerformanceResult(), 'delivery_time': MechanismPerformanceResult(), 'delivery_delay': MechanismPerformanceResult(), 'no_delivery': MechanismPerformanceResult(), 'late_delivery_flag': MechanismPerformanceResult(), 'cancellation': MechanismPerformanceResult(), 'is_dissatisfied': MechanismPerformanceResult()}, pnl_assumptions={'delivery_delay': (np.float64(0.6041446754199692), np.False_, 0.05), 'no_delivery': (np.float64(0.0), np.True_, 0.05), 'delivery_time': (np.float64(1.0), np.False_, 0.05), 'is_dissatisfied': (np.float64(0.0), np.True_, 0.05), 'late_delivery_flag': (np.float64(0.0), np.True_, 0.05), 'freight_value': (np.float64(0.8045100890308754), np.False_, 0.05), 'total_payment': (np.float64(1.0), n

In [ ]:
# BASELINE SAMPLE
baseline=(data_customer["is_dissatisfied"].mean())
print("baseline_dissatisfaction")
baseline

baseline_dissatisfaction


np.float64(0.14641434262948208)

intervensi

In [ ]:
# INTERVENSI 1
# TURUNKAN ONGKIR
int_freight = gcm.interventional_samples(
    causal_model,
    interventions={
        "freight_value": lambda x: x * 0.9
    },
    observed_data = data_customer
)

In [ ]:
print("is_dissatisfied intervention:")
print(data_customer["is_dissatisfied"].mean())

is_dissatisfied intervention:
0.14641434262948208


In [ ]:
print("after intervention")
print(int_freight["is_dissatisfied"].mean())

after intervention
0.13692598028936884


In [ ]:
comparison = pd.DataFrame({
    "Variabel": ["freight_value", "is_dissatisfied"],
    "Sebelum": [
        data_customer["freight_value"].mean(),
        data_customer["is_dissatisfied"].mean()
    ],
    "Sesudah": [
        int_freight["freight_value"].mean(),
        int_freight["is_dissatisfied"].mean()
    ]
})

comparison["Efek Intervensi"] = comparison["Sesudah"] - comparison["Sebelum"]

comparison

,Variabel,Sebelum,Sesudah,Efek Intervensi
0,freight_value,20.687375,18.613992,-2.073382
1,is_dissatisfied,0.146414,0.136926,-0.009488


In [ ]:
# INTERVENSI 2
# PERCEPAT PENGIRIMAN
# delivery time turun 10%
int_delivery = gcm.interventional_samples(
    causal_model,
    interventions={
        "delivery_time": lambda x: x * 0.9
    },
    observed_data = data_customer
)

In [ ]:
print("is_dissatisfied intervention:")
print(data_customer["is_dissatisfied"].mean())

is_dissatisfied intervention:
0.14641434262948208


In [ ]:
print("after intervention")
print(int_delivery["is_dissatisfied"].mean())

after intervention
0.11863074019710632


In [ ]:
comparison = pd.DataFrame({
    "Variabel": ["delivery_time", "is_dissatisfied"],
    "Sebelum": [
        data_customer["delivery_time"].mean(),
        data_customer["is_dissatisfied"].mean()
    ],
    "Sesudah": [
        int_delivery["delivery_time"].mean(),
        int_delivery["is_dissatisfied"].mean()
    ]
})

comparison["Efek Intervensi"] = comparison["Sesudah"] - comparison["Sebelum"]

comparison

,Variabel,Sebelum,Sesudah,Efek Intervensi
0,delivery_time,11.730914,10.570886,-1.160028
1,is_dissatisfied,0.146414,0.118631,-0.027784


In [ ]:
# INTERVENSI 3 (late_delivery_flag)
int_late = gcm.interventional_samples(
    causal_model,
    interventions={
        "late_delivery_flag": lambda x: x * 0.9
    },
    observed_data = data_customer
)

In [ ]:
print("is_dissatisfied intervention:")
print(data_customer["is_dissatisfied"].mean())

is_dissatisfied intervention:
0.14641434262948208


In [ ]:
print("after intervention")
print(int_late["is_dissatisfied"].mean())

after intervention
0.13840427762633675


In [ ]:
comparison = pd.DataFrame({
    "Variabel": ["late_delivery_flag", "is_dissatisfied"],
    "Sebelum": [
        data_customer["late_delivery_flag"].mean(),
        data_customer["is_dissatisfied"].mean()
    ],
    "Sesudah": [
        int_late["late_delivery_flag"].mean(),
        int_late["is_dissatisfied"].mean()
    ]
})

comparison["Efek Intervensi"] = comparison["Sesudah"] - comparison["Sebelum"]

comparison

,Variabel,Sebelum,Sesudah,Efek Intervensi
0,late_delivery_flag,0.065072,0.058608,-0.006464
1,is_dissatisfied,0.146414,0.138404,-0.008010


In [ ]:
# INTERVENSI 4
# Tambah deskripsi produk
int_desc = gcm.interventional_samples(
    causal_model,
    interventions={
        "product_description_length": lambda x: x + 100
    },
    observed_data = data_customer
)

In [ ]:
print("is_dissatisfied intervention:")
print(data_customer["is_dissatisfied"].mean())

is_dissatisfied intervention:
0.14641434262948208


In [ ]:
print("after intervention")
print(int_desc["is_dissatisfied"].mean())

after intervention
0.12323338226043196


In [ ]:
comparison = pd.DataFrame({
    "Variabel": ["product_description_length", "is_dissatisfied"],
    "Sebelum": [
        data_customer["product_description_length"].mean(),
        data_customer["is_dissatisfied"].mean()
    ],
    "Sesudah": [
        int_desc["product_description_length"].mean(),
        int_desc["is_dissatisfied"].mean()
    ]
})

comparison["Efek Intervensi"] = comparison["Sesudah"] - comparison["Sebelum"]

comparison

,Variabel,Sebelum,Sesudah,Efek Intervensi
0,product_description_length,791.845471,891.845471,100.000000
1,is_dissatisfied,0.146414,0.123233,-0.023181


In [ ]:
# intervensi 5
# no delivery
int_no_delivery = gcm.interventional_samples(
    causal_model,
    interventions={
        "no_delivery": lambda x: 0
    },
    observed_data = data_customer
)

In [ ]:
print("is_dissatisfied intervention:")
print(data_customer["is_dissatisfied"].mean())

is_dissatisfied intervention:
0.14641434262948208


In [ ]:
print("after intervention")
print(int_no_delivery["is_dissatisfied"].mean())

after intervention
0.11820088068777522


In [ ]:
comparison = pd.DataFrame({
    "Variabel": ["no_delivery", "is_dissatisfied"],
    "Sebelum": [
        data_customer["no_delivery"].mean(),
        data_customer["is_dissatisfied"].mean()
    ],
    "Sesudah": [
        int_no_delivery["no_delivery"].mean(),
        int_no_delivery["is_dissatisfied"].mean()
    ]
})

comparison["Efek Intervensi"] = comparison["Sesudah"] - comparison["Sebelum"]

comparison

,Variabel,Sebelum,Sesudah,Efek Intervensi
0,no_delivery,0.028511,0.000000,-0.028511
1,is_dissatisfied,0.146414,0.118201,-0.028213


In [ ]:
summary = pd.DataFrame({
    "Scenario": [
        "Baseline",
        "Freight Value ↓",
        "Delivery Time ↓",
        "Late Delivery Flag ↓",
        "Product Description ↑",
        "No delivery ↑",
    ],
    "Mean_is_dissatisfied": [
        baseline,
        int_freight["is_dissatisfied"].mean(),
        int_delivery["is_dissatisfied"].mean(),
        int_late["is_dissatisfied"].mean(),
        int_desc["is_dissatisfied"].mean(),
        int_no_delivery["is_dissatisfied"].mean(),
    ]
})

display(summary)

,Scenario,Mean_is_dissatisfied
0,Baseline,0.146414
1,Freight Value ↓,0.136926
2,Delivery Time ↓,0.118631
3,Late Delivery Flag ↓,0.138404
4,Product Description ↑,0.123233
5,No delivery ↑,0.118201
